# Data Profiling — Olist Source Tables

Profiling the nine raw CSV files before any cleaning.
Requirement 3.2: missing values, duplicates, referential integrity.

In [0]:
#lexo 9 skedaret ne nje dfs dictionary, printo rreshtat dhe kolonat
from pyspark.sql import functions as F

BASE = "/Volumes/olist/landing/files"

tables = {
    "orders":         "olist_orders_dataset.csv",
    "order_items":    "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews":  "olist_order_reviews_dataset.csv",
    "customers":      "olist_customers_dataset.csv",
    "sellers":        "olist_sellers_dataset.csv",
    "products":       "olist_products_dataset.csv",
    "geolocation":    "olist_geolocation_dataset.csv",
    "category_tr":    "product_category_name_translation.csv",
}

# Ngarkuar si string i papërpunuar — pa skemë, që të shohim gjendjen reale.
dfs = {name: spark.read.option("header", True).csv(f"{BASE}/{file}")
       for name, file in tables.items()}

for name, df in dfs.items():
    print(f"{name:16s}: {df.count():>9,} rows, {len(df.columns)} columns")

In [0]:
# Missing values per column, dallon null dhe string bosh
for name, df in dfs.items():
    print(f"\n{name}")
    cols = df.columns
    total = df.count()
    exprs = []
    for c in cols:
        is_missing = F.col(c).isNull() | (F.length(F.trim(F.col(c))) == 0)
        exprs.append(F.count(F.when(is_missing, c)).alias(c))
    row = df.select(exprs).collect()[0]
    found = False
    for c in cols:
        if row[c] > 0:
            print(f"  {c}: {row[c]:,} ({100*row[c]/total:.1f}%)")
            found = True
    if not found:
        print("  (no missing)")

In [0]:
# Duplicate records
keys = {
    "orders":        "order_id",
    "customers":     "customer_id",
    "products":      "product_id",
    "sellers":       "seller_id",
    "order_reviews": "review_id",
}
for name, key in keys.items():
    total = dfs[name].count()
    distinct = dfs[name].select(key).distinct().count()
    print(f"{name}: {total - distinct:,} duplicate {key} (of {total:,} rows)")

In [0]:
# order_reviews: a jan dublikata te verteta (pra kur rekordet jane identike) apo te njejtat id me permbajtje tjeter?
r = dfs["order_reviews"]
print("Total rows:          ", r.count())
print("Distinct review_id:  ", r.select("review_id").distinct().count())
print("Fully-unique rows:   ", r.distinct().count())

In [0]:
# Referential integrity

def orphans(child, child_key, parent, parent_key):
    c = dfs[child].select(child_key).where(F.col(child_key).isNotNull())
    p = dfs[parent].select(F.col(parent_key).alias(child_key))
    missing = c.join(p, on=child_key, how="left_anti").count()
    total = c.count()
    print(f"{child}.{child_key} -> {parent}.{parent_key}: "
          f"{missing:,} orphans of {total:,}")

orphans("order_items",    "order_id",    "orders",    "order_id")
orphans("order_payments", "order_id",    "orders",    "order_id")
orphans("order_reviews",  "order_id",    "orders",    "order_id")
orphans("order_items",    "product_id",  "products",  "product_id")
orphans("order_items",    "seller_id",   "sellers",   "seller_id")
orphans("orders",         "customer_id", "customers", "customer_id")

## Perfundime

**Missing values**
- orders: 2,965 (3%) nuk u dorezuan kurre te klienti; 1,783 pa date korrieri; 160 pa aprovim (pasi behet porosia, ose nuk kaloi pagesa ose u anulua para konfirmimit)
- order_reviews: 2,240 pa order_id, 2,380 pa score (review pa note); komentet 60-88% bosh (normale)
- products: 610 (1.9%) pa kategori dhe pershkrim; 2 rreshta pa permasa fizike
- tabelat e tjera: asnje

**Duplicates**
- order_reviews: 1,204 review_id te perseritur, por vetem 85 rreshta krejtesisht identike
- review_id nuk eshte key i besueshem vetem; key i vertete ka gjase (review_id, order_id)
- tabelat e tjera: te pastra ne primary key

**Referential Integrity**
- order_items, order_payments -> orders: 0 jetime
- order_items -> products, sellers: 0 jetime
- orders -> customers: 0 jetime
- order_reviews -> orders: 2,702 jetime — review qe referojne nje order_id qe s'ekziston te orders

**Shenim per numrin e rreshtave**
- skedari order_reviews ka 104,162 rreshta; dokumentacioni thote 99,224. Punova me te dhenat reale.

## Vendimet e pastrimit qe dalin nga keto (aplikohen ne hapat e mevonshem te silver)
- Porosi qe s'u dorezuan: mbahen, shenojme statusin e dorezimit — nuk i fshijme
- order_reviews: heqim duplikatet, 'quarantine' rreshtat me order_id null, score null, ose order_id jetim
- products: mbaj rreshtat pa kategori; perkthimi i kategorive duhet te tolerohen